# Preprocessing
## From raw prices to TDA-ready return series

Raw prices are non-stationary and unsuitable for TDA. We transform all three assets into **log returns**, handle missing data, and normalize — producing clean, aligned series ready for Takens embedding.

Assets:
- **S&P 500** (`^GSPC`) — benchmark, clean data
- **TSLA** — high volatility, artificial missing data introduced
- **BTC-USD** — cryptocurrency, different trading hours and timezone

In [ ]:
import sys
sys.path.insert(0, '/home/gabo-linux/TDA-Gabo/tda-financial-data-pipeline')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data import fetch_ticker

# Load all three assets
sp500 = fetch_ticker('^GSPC', '2000-01-01', '2024-01-01')
tsla = fetch_ticker('TSLA', '2000-01-01', '2024-01-01')
btc = fetch_ticker('BTC-USD', '2000-01-01', '2024-01-01')

# Strip timezones
sp500.index = pd.to_datetime(sp500.index, utc=True).tz_localize(None)
tsla.index = pd.to_datetime(tsla.index, utc=True).tz_localize(None)
btc.index = pd.to_datetime(btc.index, utc=True).tz_localize(None)

print("Data loaded successfully")
print(f"S&P 500: {sp500.shape} | TSLA: {tsla.shape} | BTC: {btc.shape}")

## Log Returns

Log returns are defined as:

$$r_t = \log\left(\frac{P_t}{P_{t-1}}\right)$$

Key properties:
- **Stationary** — no trend, suitable for TDA
- **Additive** — multi-period returns sum naturally
- **Symmetric** — gains and losses are treated equally
- **Standard** — universally used in quantitative finance

In [ ]:
# Compute log returns for all three assets
sp500_returns = np.log(sp500['Close'] / sp500['Close'].shift(1)).dropna()
tsla_returns = np.log(tsla['Close'] / tsla['Close'].shift(1)).dropna()
btc_returns = np.log(btc['Close'] / btc['Close'].shift(1)).dropna()

# Summary statistics
print("Log Returns Summary:")
print(f"\n{'Asset':<10} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10} {'Days':>10}")
print("-" * 55)
for name, r in [('S&P 500', sp500_returns), ('TSLA', tsla_returns), ('BTC', btc_returns)]:
    print(f"{name:<10} {r.mean():>10.6f} {r.std():>10.6f} {r.min():>10.6f} {r.max():>10.6f} {len(r):>10}")

## Observations

| Asset | Mean Daily Return | Daily Volatility | Worst Day | Best Day |
|-------|------------------|-----------------|-----------|----------|
| S&P 500 | 0.020% | 1.24% | -12.77% | +10.96% |
| TSLA | 0.149% | 3.57% | -23.65% | +21.83% |
| BTC | 0.133% | 3.73% | -46.47% | +22.51% |

BTC's worst single day loss of -46.5% reflects the extreme volatility of cryptocurrency markets. TSLA is 3x more volatile than the S&P 500 — typical for high-growth individual stocks.

## Introducing Missing Data

Yahoo Finance serves pre-cleaned data. To demonstrate missing data handling in a realistic pipeline context, we artificially introduce gaps into the TSLA series that mirror common real-world issues:

- **Trading halts** — several consecutive missing days
- **Data provider errors** — random scattered missing values

In [ ]:
np.random.seed(42)
tsla_returns_missing = tsla_returns.copy()

# Simulate trading halts — 3 blocks of 5 consecutive missing days
halt_starts = ['2013-03-15', '2016-07-20', '2021-11-10']
for start in halt_starts:
    mask = (tsla_returns_missing.index >= start)
    idx = tsla_returns_missing.index[mask][:5]
    tsla_returns_missing[idx] = np.nan

# Simulate random data provider errors — 50 random missing values
random_idx = np.random.choice(tsla_returns_missing.index, size=50, replace=False)
tsla_returns_missing[random_idx] = np.nan

print(f"Missing values introduced: {tsla_returns_missing.isnull().sum()}")
print(f"Total TSLA returns: {len(tsla_returns_missing)}")
print(f"Missing percentage: {tsla_returns_missing.isnull().sum() / len(tsla_returns_missing) * 100:.2f}%")

## Handling Missing Data

Two strategies depending on the type of missing data:

- **Random gaps** — forward fill (`ffill`) — carry last known return forward
- **Consecutive blocks** — interpolate linearly — smooth transition between known values

In practice the choice depends on the asset and the downstream task. For TDA, we prefer **linear interpolation** — it avoids introducing artificial flat segments that could create spurious topological features.

In [ ]:
# Handle missing data with linear interpolation
tsla_returns_clean = tsla_returns_missing.interpolate(method='linear')

# Verify
print(f"Missing values before: {tsla_returns_missing.isnull().sum()}")
print(f"Missing values after:  {tsla_returns_clean.isnull().sum()}")

# Visualize — compare original vs missing vs cleaned
fig, axes = plt.subplots(3, 1, figsize=(14, 10), facecolor='white', sharex=True)

axes[0].plot(tsla_returns.index, tsla_returns, color='steelblue', linewidth=0.5)
axes[0].set_title('TSLA Returns — Original')
axes[0].set_ylabel('Log Return')
axes[0].set_facecolor('white')
axes[0].grid(alpha=0.3, color='lightgrey')

axes[1].plot(tsla_returns_missing.index, tsla_returns_missing, color='red', linewidth=0.5)
axes[1].set_title('TSLA Returns — With Missing Data')
axes[1].set_ylabel('Log Return')
axes[1].set_facecolor('white')
axes[1].grid(alpha=0.3, color='lightgrey')

axes[2].plot(tsla_returns_clean.index, tsla_returns_clean, color='darkorange', linewidth=0.5)
axes[2].set_title('TSLA Returns — After Cleaning')
axes[2].set_ylabel('Log Return')
axes[2].set_facecolor('white')
axes[2].grid(alpha=0.3, color='lightgrey')

plt.tight_layout()
plt.show()

In [ ]:
# Zoom into a trading halt period
start = '2013-03-10'
end = '2013-03-25'

fig, axes = plt.subplots(3, 1, figsize=(14, 8), facecolor='white', sharex=True)

axes[0].plot(tsla_returns[start:end].index, tsla_returns[start:end], 
             color='steelblue', linewidth=1.5, marker='o', markersize=4)
axes[0].set_title('TSLA Returns — Original (zoomed)')
axes[0].set_facecolor('white')
axes[0].grid(alpha=0.3, color='lightgrey')

axes[1].plot(tsla_returns_missing[start:end].index, tsla_returns_missing[start:end], 
             color='red', linewidth=1.5, marker='o', markersize=4)
axes[1].set_title('TSLA Returns — With Missing Data (zoomed)')
axes[1].set_facecolor('white')
axes[1].grid(alpha=0.3, color='lightgrey')

axes[2].plot(tsla_returns_clean[start:end].index, tsla_returns_clean[start:end], 
             color='darkorange', linewidth=1.5, marker='o', markersize=4)
axes[2].set_title('TSLA Returns — After Cleaning (zoomed)')
axes[2].set_facecolor('white')
axes[2].grid(alpha=0.3, color='lightgrey')

plt.tight_layout()
plt.show()

## Visualization

At full scale the missing data is invisible — only 1.91% of the series. Zooming into a trading halt period reveals the gap and the interpolated values clearly.

## Normalization

We normalize all three return series to zero mean and unit variance before TDA:

$$\tilde{r}_t = \frac{r_t - \mu}{\sigma}$$

This ensures the metric space used for TDA is not dominated by scale differences between assets — BTC's higher volatility would otherwise distort the point clouds relative to S&P 500.

In [ ]:
def normalize(returns):
    return (returns - returns.mean()) / returns.std()

sp500_norm = normalize(sp500_returns)
tsla_norm = normalize(tsla_returns_clean)
btc_norm = normalize(btc_returns)

print("After normalization:")
print(f"\n{'Asset':<10} {'Mean':>10} {'Std':>10}")
print("-" * 32)
for name, r in [('S&P 500', sp500_norm), ('TSLA', tsla_norm), ('BTC', btc_norm)]:
    print(f"{name:<10} {r.mean():>10.6f} {r.std():>10.6f}")

## Observations

All three return series are now on the same scale — mean zero, unit variance. This is critical for TDA because:

- S&P 500 daily volatility was ~1.2%
- TSLA daily volatility was ~3.6%
- BTC daily volatility was ~3.7%

Without normalization, BTC and TSLA point clouds would be 3x larger than S&P 500 in the metric space — distorting topological comparisons between assets.

## Final Visualization — Normalized Returns

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), facecolor='white', sharex=True)

crises = [
    ('2000-03-01', '2002-10-01', 'red', 'Dot-com'),
    ('2008-09-01', '2009-06-01', 'orange', 'GFC'),
    ('2020-02-01', '2020-04-01', 'purple', 'COVID')
]

assets = [
    (sp500_norm, 'S&P 500 Normalized Returns', 'steelblue'),
    (tsla_norm, 'TSLA Normalized Returns', 'darkorange'),
    (btc_norm, 'BTC Normalized Returns', 'green')
]

for ax, (returns, title, color) in zip(axes, assets):
    ax.set_facecolor('white')
    ax.plot(returns.index, returns, color=color, linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.5)
    for start, end, c, label in crises:
        ax.axvspan(start, end, alpha=0.15, color=c, label=label)
    ax.set_title(title)
    ax.set_ylabel('Normalized Return')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3, color='lightgrey')

plt.tight_layout()
plt.show()

### Note on Normalization

Visually, normalized returns look similar to raw returns — normalization preserves the shape of the series, only changing the scale. The real benefit appears in Notebook 03 when all three assets are embedded into the same metric space for TDA — without normalization, BTC and TSLA point clouds would dominate S&P 500 purely due to higher volatility, not topology.

## Saving Clean Data

We save the normalized return series to disk — ready for Takens embedding in Notebook 03.

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

sp500_norm.to_csv('../data/processed/sp500_returns_normalized.csv')
tsla_norm.to_csv('../data/processed/tsla_returns_normalized.csv')
btc_norm.to_csv('../data/processed/btc_returns_normalized.csv')

print("Saved:")
print("  ../data/processed/sp500_returns_normalized.csv")
print("  ../data/processed/tsla_returns_normalized.csv")
print("  ../data/processed/btc_returns_normalized.csv")

## Conclusions

Preprocessing pipeline complete for all three assets:

| Asset | Raw Days | Clean Days | Missing Handled | Normalized |
|-------|----------|------------|-----------------|------------|
| S&P 500 | 6036 | 6036 | None needed | ✅ |
| TSLA | 3399 | 3399 | 65 interpolated | ✅ |
| BTC | 3392 | 3392 | None needed | ✅ |

All three series are now:
- **Stationary** — log returns, no trend
- **Clean** — no missing or infinite values
- **Normalized** — zero mean, unit variance
- **Saved** — cached in `data/processed/` for Notebook 03

Next: Takens embedding — converting these 1D return series into point clouds ready for TDA.